# nb00 — Colab setup + GEO download
Mount Drive, create the project folders, and acquire only the six processed paired Multiome files.
This notebook is standalone: no GitHub remote or additional source uploads are required.
Download functions below are generated from scripts/download_gse154873.py.
No biological analysis is performed. In Colab, open the notebook and run the cells in order.

In [1]:
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
    print("Local execution: using the repository root.")
else:
    IN_COLAB = True
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os

if IN_COLAB:
    BASE = Path("/content/drive/MyDrive/breast-cancer-multiome-estrogen")
else:
    BASE = Path(os.environ.get("MULTIOME_PROJECT_ROOT", Path.cwd())).resolve()
    if BASE.name == "notebooks":
        BASE = BASE.parent
    if not (BASE / "environment.yml").is_file():
        raise RuntimeError("Run locally from the repository root or set MULTIOME_PROJECT_ROOT.")

DATA_DIR = BASE / "data" / "raw" / "GSE154873"
RESULTS_DIR = BASE / "results"
for folder in (DATA_DIR, BASE / "data/processed", RESULTS_DIR / "figures", RESULTS_DIR / "tables"):
    folder.mkdir(parents=True, exist_ok=True)
print("BASE:", BASE.resolve())
print("DATA_DIR:", DATA_DIR.resolve())
print("RESULTS_DIR:", RESULTS_DIR.resolve())

BASE: /content/drive/MyDrive/breast-cancer-multiome-estrogen
DATA_DIR: /content/drive/MyDrive/breast-cancer-multiome-estrogen/data/raw/GSE154873
RESULTS_DIR: /content/drive/MyDrive/breast-cancer-multiome-estrogen/results


## Environment
Download and inspection require only Python's standard library.
The local Conda environment uses Python 3.11. Colab controls its own Python version.
Optional installation below prepares the future analysis packages; it does not run any analysis.
For exact local package versions, see requirements-lock.txt and environment-lock.linux-64.txt.

In [3]:
import sys, subprocess
print(sys.version)
INSTALL_ANALYSIS_PACKAGES = False
PACKAGES = ['scanpy', 'anndata', 'numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib', 'torch', 'jupyter', 'ipykernel', 'gseapy', 'statsmodels', 'scvi-tools']
if INSTALL_ANALYSIS_PACKAGES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


## Shared downloader
Downloads use atomic temporary files, bounded retries, expected transfer lengths, gzip CRC validation,
and SHA-256 receipts. Existing valid archives are reused; invalid archives raise a clear error.
Local SHA-256 hashes record provenance; GEO does not supply reference SHA-256 values here.
Keep DECOMPRESS=False to avoid writing approximately 6.35 GB of text. Archives are always retained.

In [4]:
#!/usr/bin/env python3
"""Download only six paired Multiome supplements; standard library only."""
import argparse
import gzip
import hashlib
import json
import shutil
import time
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

ACCESSION = "GSE154873"
BASE_URL = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{ACCESSION[:-3]}nnn/{ACCESSION}/suppl/"
FILES = {
    f"{line.lower()}_{kind.lower()}": f"{ACCESSION}_{line}_multiome_{kind}.tsv.gz"
    for line in ("MCF7", "T47D")
    for kind in ("RNA_counts", "ATAC_counts", "metadata")
}
FILES = {key.replace("_counts", ""): value for key, value in FILES.items()}
CHUNK = 1024 * 1024


def file_url(filename):
    return BASE_URL + urllib.parse.quote(filename, safe="")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(CHUNK), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_gzip(path):
    """Read to EOF to validate every gzip member/CRC, without retaining payload."""
    total = 0
    with gzip.open(path, "rb") as handle:
        for block in iter(lambda: handle.read(CHUNK), b""):
            total += len(block)
    if not total:
        raise ValueError(f"Empty gzip payload: {path}")
    return total


def fetch(url, target, attempts=3):
    """Never expose an incomplete transfer as the final filename."""
    partial = target.with_name(target.name + ".part")
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "GSE154873-reproducible-setup/1.0"})
            with urllib.request.urlopen(request, timeout=120) as response, partial.open("wb") as output:
                length = response.headers.get("Content-Length")
                expected = int(length) if length else None
                received, last = 0, 0.0
                for block in iter(lambda: response.read(CHUNK), b""):
                    output.write(block)
                    received += len(block)
                    now = time.monotonic()
                    if now - last >= 2:
                        fraction = f"/{expected / 1e6:.1f} MB ({100 * received / expected:.1f}%)" if expected else " MB"
                        print(f"  {received / 1e6:.1f}{fraction}", flush=True)
                        last = now
            if not received or (expected is not None and received != expected):
                raise IOError(f"Incomplete transfer: {received} bytes, expected {expected}")
            validate_gzip(partial)
            partial.replace(target)
            print(f"  Complete: {received:,} bytes", flush=True)
            return
        except Exception as exc:
            partial.unlink(missing_ok=True)
            if attempt == attempts:
                raise RuntimeError(f"Download failed after {attempts} attempts: {url}: {exc}") from exc
            print(f"  Attempt {attempt} failed: {exc}; retrying.", flush=True)
            time.sleep(attempt * 2)


def decompress_archive(path, expected_size, expected_hash=None):
    target = path.with_suffix("")
    if target.exists() and target.stat().st_size == expected_size and expected_hash and sha256(target) == expected_hash:
        print(f"  Decompressed file already verified: {target.stat().st_size:,} bytes")
        return expected_hash
    partial = target.with_name(target.name + ".part")
    try:
        with gzip.open(path, "rb") as source, partial.open("wb") as output:
            shutil.copyfileobj(source, output, CHUNK)
        if partial.stat().st_size != expected_size:
            raise IOError(f"Decompression size mismatch: {path}")
        digest = sha256(partial)
        partial.replace(target)
    finally:
        partial.unlink(missing_ok=True)
    print(f"  Decompressed: {expected_size:,} bytes; compressed original retained")
    return digest


def download_all(destination, decompress=False):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    manifest_path = destination / "download_manifest.json"
    previous = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    records = {}
    for key, filename in FILES.items():
        path = destination / filename
        print(f"\n{key}: {filename}", flush=True)
        if path.exists():
            print("  Already exists; validating before skipping download.", flush=True)
            try:
                expanded_size = validate_gzip(path)
                digest = sha256(path)
                old = previous.get("files", {}).get(key, {})
                if old.get("sha256") and old["sha256"] != digest:
                    raise ValueError("SHA-256 differs from the prior manifest")
            except Exception as exc:
                raise RuntimeError(f"Existing file is invalid: {path}. Move it aside and rerun. {exc}") from exc
        else:
            fetch(file_url(filename), path)
            expanded_size = validate_gzip(path)
            digest = sha256(path)
        record = dict(filename=filename, url=file_url(filename), compressed_bytes=path.stat().st_size,
                      uncompressed_bytes=expanded_size, sha256=digest, gzip_crc_valid=True)
        print(f"  Compressed: {record['compressed_bytes']:,} bytes; payload: {expanded_size:,} bytes", flush=True)
        if decompress:
            record["decompressed_sha256"] = decompress_archive(
                path, expanded_size, previous.get("files", {}).get(key, {}).get("decompressed_sha256"))
        records[key] = record
    missing = [name for name in FILES.values() if not (destination / name).is_file()]
    if missing:
        raise RuntimeError(f"Missing required files: {missing}")
    manifest = dict(accession=ACCESSION, verified_at_utc=datetime.now(timezone.utc).isoformat(), files=records)
    temp = manifest_path.with_suffix(".json.part")
    temp.write_text(json.dumps(manifest, indent=2) + "\n")
    temp.replace(manifest_path)
    print("\nAll six files verified (gzip CRC and local SHA-256).")
    return manifest


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dest", type=Path, default=Path(__file__).resolve().parents[1] / "data/raw/GSE154873")
    parser.add_argument("--decompress", action="store_true", help="Also unpack TSV files; retain .gz originals.")
    parser.add_argument("--list-urls", action="store_true", help="Print the six URLs without downloading.")
    args = parser.parse_args()
    if args.list_urls:
        for filename in FILES.values():
            print(file_url(filename))
        return
    download_all(args.dest, args.decompress)

In [5]:
DECOMPRESS = False
manifest = download_all(DATA_DIR, decompress=DECOMPRESS)


mcf7_rna: GSE154873_MCF7_multiome_RNA_counts.tsv.gz
  1.0/21.9 MB (4.8%)
  Complete: 21,850,925 bytes
  Compressed: 21,850,925 bytes; payload: 472,334,144 bytes

mcf7_atac: GSE154873_MCF7_multiome_ATAC_counts.tsv.gz
  1.0/66.8 MB (1.6%)
  52.4/66.8 MB (78.5%)
  Complete: 66,788,657 bytes
  Compressed: 66,788,657 bytes; payload: 2,659,825,102 bytes

mcf7_metadata: GSE154873_MCF7_multiome_metadata.tsv.gz
  0.0/0.0 MB (100.0%)
  Complete: 34,522 bytes
  Compressed: 34,522 bytes; payload: 411,584 bytes

t47d_rna: GSE154873_T47D_multiome_RNA_counts.tsv.gz
  1.0/23.7 MB (4.4%)
  Complete: 23,679,618 bytes
  Compressed: 23,679,618 bytes; payload: 485,405,233 bytes

t47d_atac: GSE154873_T47D_multiome_ATAC_counts.tsv.gz
  1.0/86.3 MB (1.2%)
  53.5/86.3 MB (62.0%)
  Complete: 86,265,641 bytes
  Compressed: 86,265,641 bytes; payload: 2,732,940,807 bytes

t47d_metadata: GSE154873_T47D_multiome_metadata.tsv.gz
  0.0/0.0 MB (100.0%)
  Complete: 35,325 bytes
  Compressed: 35,325 bytes; payload: 422,

In [6]:
import json
receipt = RESULTS_DIR / "tables" / "download_manifest.json"
receipt.write_text(json.dumps(manifest, indent=2) + "\n")
for key, item in manifest["files"].items():
    path = DATA_DIR / item["filename"]
    assert path.is_file() and path.stat().st_size == item["compressed_bytes"]
    print(key, f'{item["compressed_bytes"]:,} compressed bytes', item["sha256"])
print("All six files verified. Next: nb01_data_inspection.ipynb")

mcf7_rna 21,850,925 compressed bytes f9db7d5250502cceeca67c172eecb833d1c77772650636482f4560b00946915d
mcf7_atac 66,788,657 compressed bytes 80aacae2e0e84ec2268e3dc7b4450aec79e5438aeec8d9fc15ea009c236c59f4
mcf7_metadata 34,522 compressed bytes 74184cef431939af755f1555fd44d0f25b292a2e0a6ba5663ad574d5609e9a7c
t47d_rna 23,679,618 compressed bytes 576f0d067d91f8e435b4bd3fa47cb1cbc541d724293965ad629dd52047f3004f
t47d_atac 86,265,641 compressed bytes 6c72f18ef3fcf49e7a4d5971e6aa4073b4982c8c1ab8a8e9ed810e2ab954bce7
t47d_metadata 35,325 compressed bytes e36d108ab2aae925007f0d5dee10cba8a2be0308aac0570a4109f54903bca542
All six files verified. Next: nb01_data_inspection.ipynb
